In [1]:
from ipycanvas import Canvas, MultiCanvas
import ipywidgets as widgets
from IPython.display import display, HTML
import time
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from matplotlib.animation import FuncAnimation
import matplotlib

# TODO: Figure out how to utilize multi-layer canvas to reduce/prevent flickering while drawing graphs
multi_canvas = MultiCanvas(2, width=1000, height=600)
background_layer, draw_layer = multi_canvas
multi_canvas.layout = widgets.Layout(width='1000px', height='600px')

# UI elements
mode_toggle = widgets.ToggleButtons(
    options=['Edge Creation', 'Drag Vertices'],
    value='Edge Creation',
    description='Mode:',
    style={'description_width': 'initial'}
)
undo_button = widgets.Button(description="Undo", layout=widgets.Layout(width='80px'))
generate_graph_button = widgets.Button(description="Generate Graph")
clear_button = widgets.Button(description="Clear Graph", layout=widgets.Layout(width='100px'))
save_graph_dropdown = widgets.Dropdown(
    options=['G1', 'G2'],
    value='G1',
    description='Save as:',
    style={'description_width': 'initial'}
)
save_graph_button = widgets.Button(description="Save Graph", layout=widgets.Layout(width='100px'))
isomorphism_button = widgets.Button(description="Run Isomorphism", layout=widgets.Layout(width='150px'))

# Output widgets for graph visualization
output_g1 = widgets.Output()
output_g2 = widgets.Output()

# Display UI
ui = widgets.VBox([
    mode_toggle,
    undo_button,
    generate_graph_button,
    clear_button,
    save_graph_dropdown,
    save_graph_button,
    isomorphism_button,
    widgets.HBox([multi_canvas, widgets.VBox([output_g1, output_g2])])
])
display(ui)

# Graph storage
last_redraw_time = time.time()
vertices = []
edges = []
action_history = []
selected_vertex = None
dragging_vertex = None
drag_start_position = None
G1 = None
G2 = None

# Drawing functions
def draw_background():
    background_layer.clear()
    background_layer.fill_style = 'white'
    background_layer.fill_rect(int(0), int(0), int(1000), int(600))

    background_layer.stroke_style = 'black'
    background_layer.line_width = float(1)
    for (i, j) in edges:
        x1, y1 = vertices[i]
        x2, y2 = vertices[j]
        background_layer.begin_path()
        background_layer.move_to(int(x1), int(y1))
        background_layer.line_to(int(x2), int(y2))
        background_layer.stroke()

    for index, (x, y) in enumerate(vertices):
        background_layer.fill_style = 'black'
        background_layer.fill_circle(int(x), int(y), int(5))
        if index == selected_vertex:
            background_layer.stroke_style = 'blue'
            background_layer.line_width = float(2)
            background_layer.stroke_circle(int(x), int(y), int(8))


def find_nearest_vertex(x, y):
    for i, (vx, vy) in enumerate(vertices):
        if ((vx - x) ** 2 + (vy - y) ** 2) ** 0.5 < 10:
            return i
    return None

def on_mouse_down(x, y):
    global selected_vertex, dragging_vertex, drag_start_position
    nearest = find_nearest_vertex(int(x), int(y))

    if mode_toggle.value == 'Edge Creation':
        if nearest is not None:
            if selected_vertex is None:
                selected_vertex = nearest
            else:
                if selected_vertex != nearest and (selected_vertex, nearest) not in edges and (nearest, selected_vertex) not in edges:
                    edges.append((selected_vertex, nearest))
                    action_history.append(('edge', (selected_vertex, nearest)))
                selected_vertex = None
        else:
            vertices.append((int(x), int(y)))
            action_history.append(('vertex', len(vertices) - 1))
            selected_vertex = None 

    elif mode_toggle.value == 'Drag Vertices' and nearest is not None:
        dragging_vertex = nearest
        drag_start_position = vertices[dragging_vertex]

def on_mouse_move(x, y):
    global last_redraw_time
    current_time = time.time()
    if mode_toggle.value == 'Drag Vertices' and dragging_vertex is not None:
        if current_time - last_redraw_time > 0.05:  # Throttle drag function to 20 FPS
            vertices[dragging_vertex] = (int(x), int(y))
            draw_background()
            last_redraw_time = current_time

def on_mouse_up(x, y):
    global dragging_vertex, drag_start_position
    if dragging_vertex is not None:
        if drag_start_position != vertices[dragging_vertex]:
            action_history.append(('move', (dragging_vertex, drag_start_position)))
        dragging_vertex = None
        drag_start_position = None
    draw_background()

def undo_action(_):
    if not action_history:
        return
    last_action, data = action_history.pop()

    if last_action == 'vertex':
        vertices.pop(data)
        global edges
        edges = [(i, j) for (i, j) in edges if i != data and j != data]

    elif last_action == 'edge':
        if data in edges:
            edges.remove(data)

    elif last_action == 'move':
        vertex_index, original_position = data
        vertices[vertex_index] = original_position

    draw_background()

# Clear graph function
def clear_graph(_):
    global vertices, edges, action_history, selected_vertex, dragging_vertex, drag_start_position
    vertices = []
    edges = []
    action_history = []
    selected_vertex = None
    dragging_vertex = None
    drag_start_position = None
    draw_background()

# Save graph function
def save_graph(_):
    canvas_height = 600
    global G1, G2
    G = nx.Graph()
    pos = {i: (int(x), canvas_height - int(y)) for i, (x, y) in enumerate(vertices)}

    for i, (x, y) in enumerate(vertices):
        G.add_node(i, pos=(int(x), int(y)))

    for i, j in edges:
        G.add_edge(i, j)

    if save_graph_dropdown.value == 'G1':
        G1 = G
        with output_g1:
            output_g1.clear_output(wait=True)
            plt.figure(figsize=(5, 3))
            nx.draw(G1, pos, with_labels=True, node_color='lightblue', edge_color='gray')
            plt.title("Graph G1")
            plt.show()
        print("Graph saved as G1")

    elif save_graph_dropdown.value == 'G2':
        G2 = G
        with output_g2:
            output_g2.clear_output(wait=True)
            plt.figure(figsize=(5, 3))
            nx.draw(G2, pos, with_labels=True, node_color='lightcoral', edge_color='gray')
            plt.title("Graph G2")
            plt.show()
        print("Graph saved as G2")

# Isomorphism check and animation
def run_isomorphism(_):
    if G1 is None or G2 is None:
        print("Both G1 and G2 need to be defined.")
        return

    GM = nx.algorithms.isomorphism.GraphMatcher(G1, G2)
    if GM.is_isomorphic():
        print("The graphs are isomorphic.")
        
        # Get original positions from G1 and G2, inverting y to prevent a mirrored graph display
        pos_G1 = {node: (x, -y) for node, (x, y) in nx.get_node_attributes(G1, 'pos').items()}
        pos_G2 = {node: (x, -y) for node, (x, y) in nx.get_node_attributes(G2, 'pos').items()}

        # Map nodes based on isomorphic matching
        mapping = GM.mapping

        # Create target positions based on the isomorphic mapping
        target_pos = {u: pos_G2[v] for u, v in mapping.items()}

        fig, ax = plt.subplots(figsize=(10, 6))

        def update(frame):
            ax.clear()
            alpha = frame / 30.0
            interpolated_pos = {
                node: (1 - alpha) * np.array(pos_G1[node]) + alpha * np.array(target_pos[node])
                for node in G1.nodes
            }
            nx.draw(G1, interpolated_pos, with_labels=True, node_color='lightcoral', edge_color='gray', ax=ax)
            ax.set_title(f"Isomorphism Animation - Frame {frame + 1}/30")

        # Close the plot to prevent static final frame
        plt.close(fig)

        anim = FuncAnimation(fig, update, frames=30, interval=100, repeat=False)
        display(HTML(anim.to_jshtml()))
    else:
        print("The graphs are not isomorphic.")

# Event bindings
multi_canvas.on_mouse_down(on_mouse_down)
multi_canvas.on_mouse_move(on_mouse_move)
multi_canvas.on_mouse_up(on_mouse_up)
undo_button.on_click(undo_action)
generate_graph_button.on_click(lambda _: generate_graph())
clear_button.on_click(clear_graph)
save_graph_button.on_click(save_graph)
isomorphism_button.on_click(run_isomorphism)

# Initialize the drawing canvas
draw_background()
